# Supervised VIS photometry without foundation features

Version 2 starts from native, background-subtracted aperture sums calibrated on
training objects, then learns a CNN correction. Validation selects the checkpoint;
test measures generalization. The aperture baseline is reported on the same objects.


## What is being measured

Choose one target definition per run: a MER fixed-aperture label or Kron flux.
These are catalog measurements, not injection truth. Keep sky splits and object
selection fixed when comparing models. Label SNR cuts can change the selected sample.

The CNN sees four channels: background-subtracted image, SNR, center prior and
valid coverage. Native linear aperture sums bypass the normalized CNN branch.
Sum and mean pooling are equivalent up to a constant for fixed-size feature maps.


## 0. Setup (edit the knobs here)

In [ ]:
import os, sys, json, time
from pathlib import Path
import numpy as np, matplotlib.pyplot as plt
from IPython.display import Image, display
NB_DIR = Path.cwd(); sys.path.insert(0, str(NB_DIR))
from simple_regressor.config import load_config
from simple_regressor import prepare as P, train as T, evaluate as E
from simple_regressor.data import load_cache
from simple_regressor.archfig import draw_architecture

# q1_cpu_pilot.json points at this repository's Q1 images and fetched MER catalog.
# For another machine, edit configs/local.json and set this path accordingly.
CONFIG = NB_DIR / 'configs' / 'q1_cpu_pilot.json'
cfg = load_config(CONFIG)
# Use a distinct output_dir for each target/model comparison.
print(json.dumps(cfg.to_json(), indent=2))


## 1. Extract stamps  (overwrites the cache each run)

In [ ]:
P.prepare(cfg)
cache = load_cache(str(Path(cfg.output_dir)/'stamps_cache.npz')); split = cache['split'].astype(str)
snr = cache['flux']/cache['fluxerr']          # SNR of the *label* flux
print('label:', cfg.flux_column)
for s in ('train','val','test'):
    m = split==s; print(f'{s:5s}: n={m.sum():5d}  median label SNR={np.median(snr[m]):.1f}  median label frac err={np.median(1/snr[m]):.3f}')

### Example stamps and the training magnitude distribution

In [ ]:
idx = np.argsort(cache['mag'])[np.linspace(0,len(split)-1,8).astype(int)]
fig,ax=plt.subplots(1,9,figsize=(18,2.2))
for a,i in zip(ax[:8],idx):
    im=cache['stamps'][i,0].astype(np.float32); a.imshow(np.arcsinh(im/max(np.std(im),1e-3)),cmap='gray',origin='lower')
    a.set_title('mag %.1f'%cache['mag'][i],fontsize=8); a.axis('off')
ax[8].hist(cache['mag'][split=='train'],bins=25,color='0.6'); ax[8].set_title('train mag (label)',fontsize=9)
plt.tight_layout(); plt.show()

### Model architecture (drawn from the current settings)
Redrawn from `cfg` every run, so input size, binning, pooling branches, head width and label
always match what is trained. Input stamp is a real example.

In [ ]:
i_ex = int(np.argsort(np.abs(cache['mag'] - 21.0))[0])
fig = draw_architecture(cfg, stamp=cache['stamps'][i_ex, 0], path=str(Path(cfg.output_dir)/'model_architecture.png'))
plt.show()

## 2. Train  (watch train vs val)

In [ ]:
t0=time.time(); T.train(cfg); print('elapsed %.0f s'%(time.time()-t0))

### Training versus validation

Balanced sampling, when enabled, changes the training population. Use the later
unaugmented evaluation on each split for directly comparable metrics. Epoch -1
is the fitted aperture initialization and is eligible for best.pt.


In [ ]:
h=json.loads((Path(cfg.output_dir)/'history.json').read_text())['history']
ep=[r['epoch'] for r in h]
plt.figure(figsize=(7,4))
plt.plot(ep,[r['train_loss'] for r in h],label='train'); plt.plot(ep,[r['val_loss'] for r in h],label='val')
plt.yscale('log'); plt.xlabel('epoch'); plt.ylabel(f'loss ({cfg.loss})'); plt.legend()
plt.title('train vs val (gap = memorisation; both high = not extracted)'); plt.show()

## 3. Evaluate — train (fit) vs val/test (generalisation)

In [ ]:
import csv
res={}
for s in ['train','val','test']:
    res[s]=E.evaluate(cfg.output_dir, s, save_plot=(s!='train'))['overall']
print('\n%-6s %10s %10s %9s'%('split','NMADfrac','med|q|','within1s'))
for s in ['train','val','test']:
    o=res[s]; print('%-6s %10.3f %10.2f %8.1f%%'%(s,o['nmad_fractional_flux_error'],o['median_abs_q'],100*o['fraction_within_1sigma']))

# Per-magnitude: scatter vs MER's quoted error for this label. Ratio ~1 = at the quoted-error
# floor; ratio >> 1 = something other than quoted label noise (model, definition, mismatch).
nm = lambda a: 1.4826*np.median(np.abs(a-np.median(a)))
for s in ['train','test']:
    r=list(csv.DictReader(open(Path(cfg.output_dir)/f'predictions_{s}.csv')))
    t=np.array([float(x['flux_true_ujy']) for x in r]); p=np.array([float(x['flux_pred_ujy']) for x in r])
    e=np.array([float(x['fluxerr_ujy']) for x in r]); m=np.array([float(x['mag']) for x in r])
    ok=p>0; dm=-2.5*np.log10(p[ok]/t[ok]); sm=1.0857*e[ok]/t[ok]; m=m[ok]
    print(f'\n[{s}] NMAD(dmag) vs MER quoted sigma_mag  (label: {cfg.flux_column})')
    for lo,hi in [(16,22),(22,23),(23,24),(24,25),(25,26)]:
        k=(m>=lo)&(m<hi)
        if k.sum()>5: print(f'  mag {lo}-{hi}: n={k.sum():4d}  NMAD={nm(dm[k]):.3f}  quoted={np.median(sm[k]):.3f}  ratio={nm(dm[k])/np.median(sm[k]):.1f}')
display(Image(str(Path(cfg.output_dir)/'diagnostic_test.png')))

### 3b. Aperture baseline on the same held-out objects

The baseline is fitted only on training data and saved in the model checkpoint.
Evaluation reports it beside the corrected prediction. Negative baseline fluxes
remain in flux metrics; magnitude metrics exclude them and report their fraction.
Catalog-normalized residual q is not a predicted uncertainty calibration test.


In [ ]:
report = E.evaluate(cfg.output_dir, 'test', save_plot=False)
print('CNN:', report['overall'])
print('Aperture initialization:', report['aperture_baseline'])
for row in report['magnitude_bins']:
    base = row.get('aperture_baseline', {})
    print(row['mag_min'], row['mag_max'], 'n=', row['n'],
          'CNN NMAD:', row['nmad_delta_mag'],
          'aperture NMAD:', base.get('nmad_delta_mag'),
          'aperture nonpositive:', base.get('nonpositive_prediction_fraction'))


## 4. Optional pooling comparison (select on validation)

Each variant gets its own folder. Keep test data for the final selected model.


In [ ]:
COMPARE = False
if COMPARE:
    import copy, shutil
    rows = []
    for pool in ['avgmax', 'gatedsum']:
        c = copy.deepcopy(cfg); c.pool = pool
        c.output_dir = str(Path(cfg.output_dir).parent / (Path(cfg.output_dir).name + '_' + pool))
        out = Path(c.output_dir); out.mkdir(parents=True, exist_ok=True)
        for name in ['stamps_cache.npz', 'metadata.json']:
            shutil.copy2(Path(cfg.output_dir) / name, out / name)
        T.train(c)
        o = E.evaluate(c.output_dir, 'val', save_plot=False)['overall']
        rows.append((pool, o['nmad_fractional_flux_error'], o['median_abs_q']))
    print(rows)


## Interpretation

- First verify exact pixel-aperture recovery with the synthetic tests, then compare
  against MER aperture measurements; Kron introduces a different target definition.
- best.pt may remain at epoch -1 if learning fails to improve validation loss.
- Compare bias, scatter, outliers and nonpositive predictions, not only NMAD.
- Rebuild caches with version 2: invalid image/variance pixels are excluded jointly,
  and stamps are stored as float32. Variance maps are required.
- Flux uncertainties are not predicted by this baseline. q uses catalog errors only.
- The foundation comparison will later keep targets/splits fixed and test whether
  multi-band context improves measurements, especially in crowded fields.
